# 🏆 학생 건강상태 분류 — 점수 끌어올리기 (Lv4 / 앙상블)

**목표: 리더보드 상위권(1등 0.95182) 추격.** 현재 단일모델 ~0.904 → 앙상블로 개선.

> 이 노트북은 **캐글 상위권의 표준 무기**를 씁니다:
> 1. **OOF(Out-Of-Fold) 교차검증** — 누수 없이 정직한 점수 측정
> 2. **4종 부스팅 앙상블** (LightGBM · XGBoost · CatBoost · HistGradientBoosting) — 서로의 실수를 상쇄
> 3. **fold 배깅** — fold별 모델의 test 확률을 평균 → 분산↓
> 4. **early stopping** — 나무 수를 자동으로 최적화
>
> ⚠️ 앞 노트북(01_EDA, 02_모델링)은 **과정 기록으로 그대로 두세요.** 이건 별도의 '점수 짜내기' 단계입니다.

### 왜 앙상블이 오르나? (직관)
네 모델은 비슷한 점수라도 **틀리는 샘플이 서로 달라요.** 확률을 평균내면 한 모델의 실수를 다른 모델이 메워서, 단일 최고 모델보다 대체로 더 높고 안정적입니다.


## 0. 설치 & 설정

**핵심 설정(맨 위에서 조절):**
- `METRIC` / `USE_CLASS_WEIGHT`: 대회 지표에 맞춤. **balanced accuracy면 True 유지**, 혹시 **그냥 accuracy면 False**로.
- `RUN_FRACTION`: 처음엔 0.3으로 빠르게 파이프라인 검증 → 잘 돌면 **1.0으로 바꿔 최종 실행**.


In [ ]:
!pip install -q lightgbm xgboost catboost
print('설치 완료')


In [ ]:
import time, warnings, os
warnings.filterwarnings('ignore')
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import balanced_accuracy_score, f1_score, accuracy_score, classification_report
from sklearn.utils.class_weight import compute_sample_weight

# ===== 핵심 설정 =====
METRIC           = 'balanced_accuracy'   # 대회 지표. 'accuracy'이면 아래 USE_CLASS_WEIGHT=False 권장
USE_CLASS_WEIGHT = True                   # balanced accuracy면 True, 순수 accuracy면 False
RUN_FRACTION     = 0.3                    # 먼저 0.3으로 검증 → 최종엔 1.0
N_SPLITS         = 5
RS               = 42
np.random.seed(RS)
score_fn = balanced_accuracy_score if METRIC=='balanced_accuracy' else accuracy_score
print(f'METRIC={METRIC} | class_weight={USE_CLASS_WEIGHT} | fraction={RUN_FRACTION} | folds={N_SPLITS}')


## 1. 데이터 로드 + 파생변수 (self-contained)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
DATA_PATH='/content/drive/MyDrive/Colab Notebooks/2026/이어드림스쿨6기/dataset/kaggle_student_classification/'
train=pd.read_csv(DATA_PATH+'train.csv'); test=pd.read_csv(DATA_PATH+'test.csv')
TARGET,ID='health_condition','id'
numeric_features=['sleep_duration','heart_rate','bmi','calorie_expenditure','step_count','exercise_duration','water_intake']
categorical_features=['diet_type','stress_level','sleep_quality','physical_activity_level','smoking_alcohol','gender']
feature_cols=numeric_features+categorical_features
HR_HI=train['heart_rate'].quantile(0.75); STEP_LO=train['step_count'].quantile(0.25); WATER_LO=train['water_intake'].quantile(0.25)
ord_maps={'stress_level':{'low':0,'medium':1,'high':2},'sleep_quality':{'poor':0,'average':1,'good':2},
 'physical_activity_level':{'sedentary':0,'moderate':1,'active':2},'smoking_alcohol':{'no':0,'occasional':1,'yes':2}}
def make_features(df):
    X=df.copy()
    for col,m in ord_maps.items(): X[col+'_ord']=X[col].map(m)
    X['sleep_debt']=(7-X['sleep_duration']).clip(lower=0); X['sleep_excess']=(X['sleep_duration']-9).clip(lower=0)
    X['sleep_ideal']=X['sleep_duration'].between(7,9).astype('float'); X.loc[X['sleep_duration'].isna(),'sleep_ideal']=np.nan
    X['bmi_cat']=pd.cut(X['bmi'],bins=[-np.inf,18.5,25,30,np.inf],labels=[0,1,2,3]).astype('float')
    X['bmi_abnormal']=(~X['bmi'].between(18.5,25)).astype('float'); X.loc[X['bmi'].isna(),'bmi_abnormal']=np.nan
    X['hr_high']=(X['heart_rate']>HR_HI).astype('float'); X.loc[X['heart_rate'].isna(),'hr_high']=np.nan
    X['steps_per_ex_min']=X['step_count']/(X['exercise_duration']+1); X['cal_per_step']=X['calorie_expenditure']/(X['step_count']+1)
    X['low_activity']=(X['step_count']<STEP_LO).astype('float'); X.loc[X['step_count'].isna(),'low_activity']=np.nan
    X['low_water']=(X['water_intake']<WATER_LO).astype('float'); X.loc[X['water_intake'].isna(),'low_water']=np.nan
    risk=pd.DataFrame(index=X.index)
    risk['a']=(X['sleep_duration']<6).astype(float); risk['b']=(X['stress_level']=='high').astype(float)
    risk['c']=(X['sleep_quality']=='poor').astype(float); risk['d']=(X['physical_activity_level']=='sedentary').astype(float)
    risk['e']=(X['step_count']<STEP_LO).astype(float); risk['f']=(X['smoking_alcohol']=='yes').astype(float)
    risk['g']=(~X['bmi'].between(18.5,25)).astype(float)
    X['lifestyle_risk_score']=risk.sum(axis=1); X['n_missing']=df[feature_cols].isnull().sum(axis=1)
    for c in feature_cols: X[c+'_isna']=df[c].isnull().astype('int8')
    return X
train_fe=make_features(train); test_fe=make_features(test)

# 명목형(diet_type, gender)을 숫자 코드로 (트리는 NaN 그대로 처리 가능)
for c in ['diet_type','gender']:
    cats=train_fe[c].astype('category').cat.categories
    mp={k:i for i,k in enumerate(cats)}
    train_fe[c+'_code']=train_fe[c].map(mp); test_fe[c+'_code']=test_fe[c].map(mp)
print('train_fe', train_fe.shape)


## 2. 두 가지 변수 세트 (가지치기 진단용)

- `FEATURES_PRUNED`: 02 노트북에서 쓴 정예 세트
- `FEATURES_FULL`: 죽은 변수까지 **전부** 포함

EDA 빠른 테스트(0.947)와 실전(0.904)의 격차 원인을 확인하려고, **둘을 같은 조건에서 비교**합니다.
부스팅은 결측을 그대로 학습하므로 **대치·스케일 없이** 넣습니다.


In [ ]:
FEATURES_PRUNED = numeric_features + [
    'lifestyle_risk_score','sleep_debt','sleep_ideal','bmi_cat','cal_per_step','steps_per_ex_min','low_activity',
    'stress_level_ord','sleep_quality_ord','physical_activity_level_ord','smoking_alcohol_ord',
    'diet_type_code','gender_code']
exclude = [ID, TARGET] + categorical_features
FEATURES_FULL = [c for c in train_fe.columns if c not in exclude and train_fe[c].dtype != 'object']
print('PRUNED', len(FEATURES_PRUNED), '| FULL', len(FEATURES_FULL))

# 라벨 인코딩 + 실행 샘플
le = LabelEncoder().fit(train_fe[TARGET])
CLASSES = le.classes_
if RUN_FRACTION < 1.0:
    run = train_fe.groupby(TARGET, group_keys=False).apply(lambda s: s.sample(frac=RUN_FRACTION, random_state=RS))
else:
    run = train_fe
y_run = le.transform(run[TARGET])
print('실행 데이터:', run.shape, '| 클래스:', dict(zip(CLASSES, np.bincount(y_run))))


## 3. OOF 학습 함수 (핵심 엔진)

한 모델을 K-fold로 학습하면서:
- 각 fold의 검증부분 확률 → **OOF 예측**(정직한 CV 점수 계산용)
- 각 fold 모델의 **test 확률을 평균** → 배깅된 test 예측
- **early stopping**으로 나무 수 자동 결정

네 모델(LGBM/XGB/CatBoost/HGB)의 early stopping API가 조금씩 달라 분기 처리합니다.


In [ ]:
def make_model(name):
    if name=='LightGBM':
        from lightgbm import LGBMClassifier
        return LGBMClassifier(n_estimators=3000, learning_rate=0.03, num_leaves=63,
                              subsample=0.8, colsample_bytree=0.8, reg_lambda=1.0,
                              random_state=RS, n_jobs=-1, verbose=-1)
    if name=='XGBoost':
        from xgboost import XGBClassifier
        return XGBClassifier(n_estimators=3000, learning_rate=0.03, max_depth=6,
                             subsample=0.8, colsample_bytree=0.8, reg_lambda=1.0, tree_method='hist',
                             early_stopping_rounds=100, eval_metric='mlogloss',
                             random_state=RS, n_jobs=-1, verbosity=0)
    if name=='CatBoost':
        from catboost import CatBoostClassifier
        return CatBoostClassifier(iterations=3000, learning_rate=0.03, depth=6, l2_leaf_reg=3.0,
                                  random_state=RS, verbose=0)
    if name=='HistGB':
        from sklearn.ensemble import HistGradientBoostingClassifier
        return HistGradientBoostingClassifier(max_iter=3000, learning_rate=0.03, max_leaf_nodes=63,
                                              l2_regularization=1.0, early_stopping=True,
                                              validation_fraction=0.1, n_iter_no_change=100, random_state=RS)

def fit_one(name, model, Xtr, ytr, Xva, yva, sw):
    if name=='LightGBM':
        from lightgbm import early_stopping, log_evaluation
        model.fit(Xtr, ytr, sample_weight=sw, eval_set=[(Xva,yva)],
                  callbacks=[early_stopping(100, verbose=False), log_evaluation(0)])
    elif name=='XGBoost':
        model.fit(Xtr, ytr, sample_weight=sw, eval_set=[(Xva,yva)], verbose=False)
    elif name=='CatBoost':
        model.fit(Xtr, ytr, sample_weight=sw, eval_set=(Xva,yva), early_stopping_rounds=100, verbose=0)
    else:  # HistGB: 내장 early stopping
        model.fit(Xtr, ytr, sample_weight=sw)
    return model

def run_oof(name, features):
    X = run[features]; Xtest = test_fe[features]
    n_cls = len(CLASSES)
    oof = np.zeros((len(X), n_cls)); test_proba = np.zeros((len(Xtest), n_cls))
    skf = StratifiedKFold(N_SPLITS, shuffle=True, random_state=RS)
    t0 = time.time()
    for fold,(tr,va) in enumerate(skf.split(X, y_run)):
        Xtr,Xva = X.iloc[tr], X.iloc[va]; ytr,yva = y_run[tr], y_run[va]
        sw = compute_sample_weight('balanced', ytr) if USE_CLASS_WEIGHT else None
        m = fit_one(name, make_model(name), Xtr, ytr, Xva, yva, sw)
        oof[va] = m.predict_proba(Xva)
        test_proba += m.predict_proba(Xtest) / N_SPLITS
    sc = score_fn(y_run, oof.argmax(1))
    print(f'  {name:10s} OOF {METRIC}={sc:.4f}  ({time.time()-t0:.0f}s)')
    return oof, test_proba, sc


## 4. 가지치기 진단 (PRUNED vs FULL)

LightGBM 하나로 두 세트를 OOF 비교합니다. **점수 높은 쪽을 최종 변수 세트로 채택.**
(0.947 vs 0.904 격차가 변수 때문인지 여기서 판가름)


In [ ]:
print('▶ 가지치기 진단 (LightGBM OOF)')
diag = {}
for tag, feats in [('PRUNED', FEATURES_PRUNED), ('FULL', FEATURES_FULL)]:
    _,_,sc = run_oof('LightGBM', feats); diag[tag]=sc
BEST_FEATURES = FEATURES_FULL if diag['FULL'] >= diag['PRUNED'] else FEATURES_PRUNED
best_tag = 'FULL' if diag['FULL']>=diag['PRUNED'] else 'PRUNED'
print(f"\n채택: {best_tag}  (PRUNED={diag['PRUNED']:.4f} vs FULL={diag['FULL']:.4f})")
print('→ 두 값이 비슷하면 변수 탓이 아니라 이전의 소표본 낙관이었다는 뜻. 큰 차이면 가지치기가 원인.')


## 5. ⭐ 4종 부스팅 OOF 학습 + 앙상블

채택된 변수 세트로 네 모델을 각각 OOF 학습하고, **test 확률을 평균**해 앙상블합니다.
⏳ RUN_FRACTION=1.0이면 오래 걸립니다(수십 분 가능). 먼저 0.3으로 검증하세요.


In [ ]:
MODELS = ['LightGBM','XGBoost','CatBoost','HistGB']
oofs, tests, scores = {}, {}, {}
print('▶ 4종 OOF 학습')
for name in MODELS:
    try:
        o,t,s = run_oof(name, BEST_FEATURES)
        oofs[name],tests[name],scores[name] = o,t,s
    except Exception as e:
        print(f'  {name} 제외: {str(e)[:70]}')

# 앙상블: OOF 확률 평균 → 점수 / test 확률 평균 → 제출용
ok = list(oofs.keys())
oof_ens  = np.mean([oofs[n]  for n in ok], axis=0)
test_ens = np.mean([tests[n] for n in ok], axis=0)
ens_score = score_fn(y_run, oof_ens.argmax(1))

tbl = pd.DataFrame({'model':ok+['🏆 앙상블'],
                    METRIC:[scores[n] for n in ok]+[ens_score]}).sort_values(METRIC, ascending=False)
display(tbl.round(4))
print(f'\n앙상블 {METRIC} = {ens_score:.4f}  (단일 최고 = {max(scores.values()):.4f})')


In [ ]:
# 앙상블 상세 리포트 (OOF 기준)
print(classification_report(y_run, oof_ens.argmax(1), target_names=CLASSES))


## 6. 제출 파일 생성

앙상블의 배깅된 test 확률 → argmax → 라벨 복원. (fold 배깅이 이미 반영됨)


In [ ]:
pred = le.inverse_transform(test_ens.argmax(1))
submission = pd.DataFrame({ID: test[ID], TARGET: pred})
tag = 'full' if RUN_FRACTION>=1.0 else f'frac{RUN_FRACTION}'
path = DATA_PATH + f'submission_ensemble_{tag}.csv'
submission.to_csv(path, index=False)
print('저장:', path, submission.shape)
print(submission[TARGET].value_counts(normalize=True).round(3).to_dict())
submission.head()


## 7. (선택) 지표가 balanced accuracy일 때 — 앙상블 가중치 탐색

단순 평균 대신 **모델별 가중치**를 조금 조정하면 OOF 점수가 더 오를 수 있어요.
간단한 랜덤 서치로 최적 가중치를 찾습니다(OOF 기준).


In [ ]:
best_w, best_s = None, ens_score
rng = np.random.RandomState(RS)
for _ in range(3000):
    w = rng.dirichlet(np.ones(len(ok)))               # 합=1 랜덤 가중치
    blend = np.tensordot(w, np.array([oofs[n] for n in ok]), axes=(0,0))
    s = score_fn(y_run, blend.argmax(1))
    if s > best_s: best_s, best_w = s, w
if best_w is not None:
    print('가중치 탐색 결과:', {n:round(float(x),3) for n,x in zip(ok,best_w)}, f'| OOF {METRIC}={best_s:.4f} (+{best_s-ens_score:.4f})')
    test_ens_w = np.tensordot(best_w, np.array([tests[n] for n in ok]), axes=(0,0))
    sub_w = pd.DataFrame({ID: test[ID], TARGET: le.inverse_transform(test_ens_w.argmax(1))})
    sub_w.to_csv(DATA_PATH + f'submission_ensemble_weighted_{tag}.csv', index=False)
    print('가중 앙상블 저장 완료')
else:
    print('단순 평균이 이미 최적 (가중치 개선 없음)')


## ✅ 정리 & 다음 수

### 점수를 올린 장치 (발표용)
1. **OOF 교차검증** — 누수 없는 정직한 점수
2. **4종 부스팅 앙상블** — 서로의 오차 상쇄
3. **fold 배깅** — test 확률 평균으로 분산↓
4. **early stopping** — 나무 수 자동 최적화
5. (선택) **가중치 탐색** — 앙상블 미세조정

### 실행 순서
1. `RUN_FRACTION=0.3`으로 전체 흐름 확인 (빠름)
2. 잘 돌면 `RUN_FRACTION=1.0`으로 최종 실행 → `submission_ensemble_full.csv` 제출

### 그래도 1등에 못 미치면 (상위권의 추가 무기)
- **하이퍼파라미터 심화 튜닝**(Optuna) — 각 부스팅을 더 짜냄
- **스태킹**(메타모델) — 단순 평균 대신 2단 학습
- **의사라벨링(pseudo-labeling)** — test 예측 중 확신 높은 것을 학습에 추가
- **피처 상호작용** — 상위 변수 간 조합 파생

> 먼저 `RUN_FRACTION=1.0` 앙상블 점수를 제출해서 리더보드가 얼마 나오는지 보고, 그 결과로 다음 수를 정하는 게 좋아요.
